# Prompt Guard: Milestone 2, data fix with short prompts

**The problem.** With the harmless prompts, our models let **15 to 20 of 60 warn prompts** through (names and
e-mail addresses in short requests). The reason is the writing style of the training data:

| Data | Decision | Median length |
|---|---|---|
| ai4privacy | warn / block | ~190 characters, formal documents |
| harmless prompts | allow | ~70 characters, short requests |
| our own test prompts | all three | ~75 characters, short requests |

So the model learned the shortcut **"short request = allow"**: it never saw a short request with a name or an IBAN.

**The fix we test here.** `src/short_prompts.py` builds short requests in German and English for **all three
classes**, with the same sentence frames ("Schreib eine Mail an {recipient} ..."): the frame is the same, only
the content of the slot decides (*"das Vertriebsteam"* = allow, *"anna.berg@example.com"* = warn,
*"IBAN DE..."* = block). The decision is calculated with our policy (`prompt_decision`), not set by hand.

**What changes:** only the training data (plus 2,000 short prompts per class and language).

**What stays the same:** split, policy, TF-IDF settings, `C` of both models (from `tfidf_harmless.ipynb` and
`svm_linear.ipynb`), the rule for choosing the threshold, the metrics and their MLflow names.

**How we measure.** Before = the models with harmless data (`tfidf_logreg_harmless`, `svm_harmless`),
after = the same models with the short prompts added. We compare them on:
1. ai4privacy validation: is recall block still ≥ 0.95?
2. harmless validation: do real harmless prompts still get through?
3. **short-prompt validation**: short requests built from *other* sentence frames than in training, so a good result
   means the model learned the content, not the frames;
4. our 200 own German prompts: above all "warn prompts let through".

**Rules:** train only on train, choose the threshold on validation, never touch a test part (also not the short-prompt
test part), only look at our own prompts.

**Structure**
1. Setup · 2. Load the data · 3. Helper functions · 4. Turn the texts into numbers · 5. TF-IDF + LogReg ·
6. Linear SVM · 7. Before vs. after · 8. Own prompts the new models get wrong · 9. Conclusion

**Before the first run:** `python src/harmless_data.py` and `python src/short_prompts.py` (fingerprints in the
README). `tfidf_harmless.ipynb` and `svm_linear.ipynb` must have been run once (their models are the "before").
With 8 GB RAM, run only this notebook and close the others.

## 1. Setup

Same settings as in the other notebooks. New is the import of `src/short_prompts.py`.

In [1]:
import os
import sys
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import joblib
import mlflow

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score, recall_score

sys.path.append("../src")
from data_utils import load_all_splits, split_fingerprint
from harmless_data import load_harmless, harmless_fingerprint
from short_prompts import load_short_prompts, short_prompts_fingerprint

SEED = 42
CLASSES = ["allow", "warn", "block"]
PROJECT_DIR = Path("..").resolve()
MODEL_DIR = PROJECT_DIR / "models"

TARGET_RECALL_BLOCK = 0.95       # at least 95 % of block texts must be blocked
TARGET_FALSE_BLOCK = 0.05        # at most 5 % of harmless (allow) texts may be blocked

TFIDF_PARAMS = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=5, max_features=300_000,
                    sublinear_tf=True, dtype=np.float32)

with open(PROJECT_DIR / "config" / "policy.yaml", encoding="utf-8") as f:
    POLICY_VERSION = str(yaml.safe_load(f)["version"])

BLUE = "#2a78d6"
ORANGE = "#eb6834"
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# MLflow: shared DagsHub server if MLFLOW_TRACKING_URI is set in .env, otherwise local
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_DIR / ".env")
except ImportError:
    pass
if os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    if mlflow.get_experiment_by_name("promptguard") is None:
        mlflow.create_experiment("promptguard")
else:
    mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_DIR / 'mlflow.db').as_posix()}")
    if mlflow.get_experiment_by_name("promptguard") is None:
        mlflow.create_experiment("promptguard", artifact_location=(PROJECT_DIR / "mlruns").as_uri())
mlflow.set_experiment("promptguard")
print("Policy version:", POLICY_VERSION, "| MLflow:", mlflow.get_tracking_uri())

c:\Users\olga9\Desktop\Studium Heilbronn\Kurse\WiSe26\AIO - AI Operations\AIOps_PromptGuard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Policy version: 0.1 | MLflow: https://dagshub.com/annihlj/AIOps_PromptGuard.mlflow


## 2. Load the data

Three sources, one table, with a column `origin` so we always know where a text comes from:
ai4privacy, harmless prompts and the new short prompts. We never load a test part.

The second table shows the point of the data fix: **after adding the short prompts, every class has short
texts**, so the length no longer tells the decision.

In [2]:
splits = load_all_splits()
pii_train, pii_val = splits["train"], splits["validation"]
del splits                                           # no test set in this notebook

SPLIT_FINGERPRINT = split_fingerprint()
HARMLESS_FINGERPRINT = harmless_fingerprint()
SHORT_FINGERPRINT = short_prompts_fingerprint()
print("Split fingerprint:   ", SPLIT_FINGERPRINT)
print("Harmless fingerprint:", HARMLESS_FINGERPRINT)
print("Short fingerprint:   ", SHORT_FINGERPRINT)

COLUMNS = ["source_text", "decision", "language"]

def tag(df, origin):
    # keep only the columns we need and add where the text comes from
    return df[COLUMNS].assign(origin=origin)

train = pd.concat([tag(pii_train, "ai4privacy"),
                   tag(load_harmless("train"), "harmless"),
                   tag(load_short_prompts("train"), "short")], ignore_index=True)
val = pd.concat([tag(pii_val, "ai4privacy"),
                 tag(load_harmless("validation"), "harmless"),
                 tag(load_short_prompts("validation"), "short")], ignore_index=True)
own = pd.read_csv(PROJECT_DIR / "data" / "own_prompts.csv")

del pii_train, pii_val                               # the full tables are no longer needed
_ = gc.collect()

is_pii = (val["origin"] == "ai4privacy").to_numpy()
is_harmless = (val["origin"] == "harmless").to_numpy()
is_short = (val["origin"] == "short").to_numpy()
is_de = (val["language"] == "de").to_numpy()
# "classic" validation = ai4privacy + harmless, the same set as in tfidf_harmless / svm_linear,
# so the logged val_* metrics stay comparable with the earlier runs
is_classic = is_pii | is_harmless

display(pd.crosstab(train["origin"], train["decision"], margins=True).rename_axis("TRAIN"))
display(pd.crosstab(val["origin"], val["decision"], margins=True).rename_axis("VALIDATION"))

lengths = train.assign(chars=train["source_text"].str.len())
display(lengths.pivot_table(index="origin", columns="decision", values="chars", aggfunc="median")
        .round(0).rename_axis("TRAIN: median length in characters"))

Split fingerprint:    6afd6a257de1
Harmless fingerprint: 32d24c900fd7
Short fingerprint:    bb97fd4d3e90


decision,allow,block,warn,All
TRAIN,,,,
ai4privacy,11471,46056,88928,146455
harmless,3864,0,0,3864
short,4000,4000,4000,12000
All,19335,50056,92928,162319


decision,allow,block,warn,All
VALIDATION,,,,
ai4privacy,2457,9868,19055,31380
harmless,557,0,0,557
short,200,200,200,600
All,3214,10068,19255,32537


decision,allow,block,warn
TRAIN: median length in characters,,,
ai4privacy,168.0,188.0,181.0
harmless,65.0,NaN,NaN
short,67.0,64.0,66.0


## 3. Helper functions

The same functions as in `svm_linear.ipynb`: they work for both model types (probabilities for Logistic
Regression, scores for the SVM), so both are measured with exactly the same code.

In [3]:
def model_scores(pipe, texts):
    # SVM -> decision_function (scores), Logistic Regression -> predict_proba (probabilities)
    if hasattr(pipe, "decision_function") and not hasattr(pipe, "predict_proba"):
        return pipe.decision_function(texts)
    return pipe.predict_proba(texts)


def predict_with_threshold(scores, classes, threshold):
    classes = list(classes)
    if threshold is None:                      # default rule: highest score wins
        return np.array(classes)[scores.argmax(axis=1)]
    s_block = scores[:, classes.index("block")]
    other = [c for c in classes if c != "block"]
    best_other = np.array(other)[scores[:, [classes.index(c) for c in other]].argmax(axis=1)]
    return np.where(s_block >= threshold, "block", best_other)


def evaluate(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    is_allow, is_warn = y_true == "allow", y_true == "warn"
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "recall_block": recall_score(y_true, y_pred, labels=["block"], average="macro", zero_division=0),
        "recall_warn": recall_score(y_true, y_pred, labels=["warn"], average="macro", zero_division=0),
        "recall_allow": recall_score(y_true, y_pred, labels=["allow"], average="macro", zero_division=0),
        "false_block_allow": float((y_pred[is_allow] == "block").mean()) if is_allow.any() else np.nan,
        "false_block_warn": float((y_pred[is_warn] == "block").mean()) if is_warn.any() else np.nan,
    }


def threshold_sweep(scores, classes, y_true, thresholds):
    # one table row per threshold, measured on ALL validation rows (ai4privacy + harmless + short prompts)
    rows = []
    for t in thresholds:
        pred = predict_with_threshold(scores, classes, t)
        rows.append({"threshold": t, **evaluate(y_true, pred)})
    sweep = pd.DataFrame(rows)
    sweep["meets both targets"] = ((sweep["recall_block"] >= TARGET_RECALL_BLOCK)
                                   & (sweep["false_block_allow"] <= TARGET_FALSE_BLOCK))
    return sweep


def choose_threshold(sweep):
    # The same rule as in all earlier notebooks, written down before looking at the results:
    # 1. highest threshold that reaches both targets, otherwise
    # 2. best recall block among thresholds with <= 5 % harmless texts blocked
    both = sweep[sweep["meets both targets"]]
    if len(both):
        return float(both["threshold"].max()), "highest threshold that reaches both targets"
    safe = sweep[sweep["false_block_allow"] <= TARGET_FALSE_BLOCK]
    if len(safe):
        return float(safe.loc[safe["recall_block"].idxmax(), "threshold"]), \
            "both targets not reachable; best recall block with <= 5 % false alarms"
    return None, "no threshold keeps false alarms <= 5 %; keep the default rule"


def fit_quietly(model, X, y):
    # trains the model and returns whether it converged (instead of printing a red warning)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        start = time.perf_counter()
        model.fit(X, y)
        seconds = time.perf_counter() - start
    return not any(issubclass(w.category, ConvergenceWarning) for w in caught), seconds


def load_bundle(filename):
    path = MODEL_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"models/{filename} is missing. Run the notebook that creates it first "
                                "(tfidf_harmless.ipynb or svm_linear.ipynb).")
    return joblib.load(path)


# Thresholds we try: probabilities for LogReg, scores for the SVM (same lists as before)
THRESHOLDS_PROBA = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10, 0.05]
THRESHOLDS_SCORE = [0.5, 0.4, 0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -1.0]

## 4. Turn the texts into numbers

Same TF-IDF settings as always, fitted once on the new train set and used by **both** models (TF-IDF + LogReg and
SVM always used the same vectorizer settings, so sharing it saves time and memory without changing anything).
We also read the `C` of both models from the "before" models, so the only change is the training data.

In [4]:
before_tfidf = load_bundle("tfidf_logreg_harmless.joblib")
before_svm = load_bundle("svm_harmless.joblib")
C_LOGREG = float(before_tfidf["pipeline"].named_steps["logreg"].C)
C_SVM = float(before_svm["pipeline"].named_steps["svm"].C)
print(f"C from the before models: LogReg {C_LOGREG}, SVM {C_SVM}")

vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
start = time.perf_counter()
X_train = vectorizer.fit_transform(train["source_text"])
X_val = vectorizer.transform(val["source_text"])
print(f"Vectorising took {time.perf_counter() - start:.0f} s, matrix: {X_train.shape}")

C from the before models: LogReg 30.0, SVM 1.0
Vectorising took 111 s, matrix: (162319, 300000)


## 5. TF-IDF + Logistic Regression with short prompts

Train, choose the block threshold on all validation rows (so the threshold also sees short requests with personal
data), save as `models/tfidf_logreg_short.joblib`.

In [5]:
logreg = LogisticRegression(C=C_LOGREG, class_weight="balanced", max_iter=3000, random_state=SEED)
converged_lr, seconds = fit_quietly(logreg, X_train, train["decision"])
print(f"Training took {seconds:.0f} s, converged: {converged_lr}")

proba_lr = logreg.predict_proba(X_val)
sweep_lr = threshold_sweep(proba_lr, logreg.classes_, val["decision"], THRESHOLDS_PROBA)
display(sweep_lr.round(4))
THRESHOLD_LR, reason_lr = choose_threshold(sweep_lr)
print(f"Chosen threshold: {THRESHOLD_LR} ({reason_lr})")

pipe_lr = Pipeline([("tfidf", vectorizer), ("logreg", logreg)])
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump({"pipeline": pipe_lr, "block_threshold": THRESHOLD_LR, "classes": list(logreg.classes_),
             "split_fingerprint": SPLIT_FINGERPRINT, "harmless_fingerprint": HARMLESS_FINGERPRINT,
             "short_prompts_fingerprint": SHORT_FINGERPRINT, "policy_version": POLICY_VERSION},
            MODEL_DIR / "tfidf_logreg_short.joblib")
print("Saved: models/tfidf_logreg_short.joblib")

Training took 178 s, converged: True


,threshold,macro_f1,recall_block,recall_warn,recall_allow,false_block_allow,false_block_warn,meets both targets
0,0.50,0.8816,0.9159,0.9160,0.8581,0.0124,0.0398,False
1,0.45,0.8800,0.9214,0.9102,0.8569,0.0143,0.0456,False
2,0.40,0.8785,0.9271,0.9039,0.8553,0.0162,0.0522,False
3,0.35,0.8762,0.9338,0.8953,0.8544,0.0184,0.0610,False
4,0.30,0.8738,0.9399,0.8869,0.8531,0.0199,0.0696,False
5,0.25,0.8708,0.9474,0.8767,0.8510,0.0233,0.0801,False
6,0.20,0.8659,0.9548,0.8626,0.8482,0.0277,0.0945,True
7,0.15,0.8594,0.9622,0.8457,0.8432,0.0352,0.1119,True
8,0.10,0.8467,0.9707,0.8152,0.8351,0.0467,0.1434,True
9,0.05,0.8195,0.9815,0.7533,0.8155,0.0737,0.2082,False


Chosen threshold: 0.2 (highest threshold that reaches both targets)
Saved: models/tfidf_logreg_short.joblib


## 6. Linear SVM with short prompts

Same as chapter 5 with the SVM (`C` from `svm_linear.ipynb`, threshold on the score scale).

In [6]:
svm = LinearSVC(C=C_SVM, class_weight="balanced", max_iter=5000, random_state=SEED)
converged_svm, seconds = fit_quietly(svm, X_train, train["decision"])
print(f"Training took {seconds:.0f} s, converged: {converged_svm}")

scores_svm = svm.decision_function(X_val)
sweep_svm = threshold_sweep(scores_svm, svm.classes_, val["decision"], THRESHOLDS_SCORE)
display(sweep_svm.round(4))
THRESHOLD_SVM, reason_svm = choose_threshold(sweep_svm)
print(f"Chosen threshold: {THRESHOLD_SVM} ({reason_svm})")

pipe_svm = Pipeline([("tfidf", vectorizer), ("svm", svm)])
joblib.dump({"pipeline": pipe_svm, "block_threshold": THRESHOLD_SVM, "score": "decision_function",
             "classes": list(svm.classes_), "split_fingerprint": SPLIT_FINGERPRINT,
             "harmless_fingerprint": HARMLESS_FINGERPRINT, "short_prompts_fingerprint": SHORT_FINGERPRINT,
             "policy_version": POLICY_VERSION}, MODEL_DIR / "svm_short.joblib")
print("Saved: models/svm_short.joblib")

# the training matrix is no longer needed
del X_train, X_val, proba_lr, scores_svm
_ = gc.collect()

Training took 38 s, converged: True


,threshold,macro_f1,recall_block,recall_warn,recall_allow,false_block_allow,false_block_warn,meets both targets
0,0.5,0.8622,0.7947,0.9592,0.8348,0.0053,0.0044,False
1,0.4,0.8704,0.8228,0.9573,0.8332,0.0068,0.0065,False
2,0.3,0.8759,0.8447,0.9549,0.8323,0.0081,0.0090,False
3,0.2,0.8813,0.8672,0.9518,0.8304,0.0103,0.0125,False
4,0.1,0.8856,0.8881,0.9474,0.8286,0.0121,0.0171,False
5,0.0,0.8880,0.9063,0.9420,0.8261,0.0146,0.0228,False
6,-0.1,0.8877,0.9196,0.9341,0.8223,0.0190,0.0313,False
7,-0.2,0.8864,0.9343,0.9243,0.8167,0.0249,0.0415,False
8,-0.3,0.8812,0.9470,0.9093,0.8071,0.0355,0.0576,False
9,-0.4,0.8744,0.9572,0.8915,0.7987,0.0442,0.0766,True


Chosen threshold: -0.4 (highest threshold that reaches both targets)
Saved: models/svm_short.joblib


## 7. Before vs. after

All four models with the same code on the same data. "Before" = with harmless data, "after" = with harmless data
and short prompts. Response time: 1,000 single prompts per model on this laptop.

The block **"short-prompt validation"** is the new check: short requests from sentence frames the models never saw.
`short val: warn let through` is the share of warn requests that got allow, the problem we want to fix.

The new runs are logged to MLflow with **the same metric names** as the earlier candidates
(`val_*` = ai4privacy + harmless validation, as in `tfidf_harmless` and `svm_linear`). The short-prompt validation
numbers are only in this table and in `reports/short_prompts_before_after.csv`.

In [7]:
MODELS = {
    "TF-IDF + harmless (before)": "tfidf_logreg_harmless.joblib",
    "TF-IDF + short prompts (after)": "tfidf_logreg_short.joblib",
    "SVM + harmless (before)": "svm_harmless.joblib",
    "SVM + short prompts (after)": "svm_short.joblib",
}
expected = own["expected_decision"].to_numpy()
casual = (own["style"] == "casual").to_numpy()
latency_sample = val.loc[is_classic, "source_text"].sample(1000, random_state=SEED).tolist()


def share(mask_true, pred, value):
    # e.g. share of true warn texts predicted as "allow"
    return float((pred[mask_true] == value).mean())


columns, metrics_for_mlflow, own_predictions = {}, {}, {}
for name, filename in MODELS.items():
    bundle = load_bundle(filename)
    pipe, threshold = bundle["pipeline"], bundle["block_threshold"]
    classes = pipe.classes_
    val_pred = predict_with_threshold(model_scores(pipe, val["source_text"]), classes, threshold)
    own_pred = predict_with_threshold(model_scores(pipe, own["prompt"].tolist()), classes, threshold)
    own_predictions[name] = own_pred

    times = []
    for prompt in latency_sample:
        start = time.perf_counter()
        predict_with_threshold(model_scores(pipe, [prompt]), classes, threshold)
        times.append((time.perf_counter() - start) * 1000)

    y = val["decision"].to_numpy()
    pii = evaluate(y[is_pii], val_pred[is_pii])
    req = evaluate(y[is_short], val_pred[is_short])
    own_block, own_warn, own_allow = expected == "block", expected == "warn", expected == "allow"
    columns[name] = {
        "block threshold": threshold,
        "ai4privacy recall block": round(pii["recall_block"], 3),
        "ai4privacy recall block (de)": round(recall_score(y[is_pii & is_de], val_pred[is_pii & is_de],
                                                           labels=["block"], average="macro"), 3),
        "ai4privacy allow texts blocked": round(pii["false_block_allow"], 3),
        "real harmless prompts allowed": f"{share(is_harmless, val_pred, 'allow'):.1%}",
        "real harmless prompts blocked": f"{share(is_harmless, val_pred, 'block'):.1%}",
        "short val: correct": f"{(val_pred[is_short] == y[is_short]).mean():.1%}",
        "short val: block caught": f"{req['recall_block']:.1%}",
        "short val: warn let through (allow)": f"{share(is_short & (y == 'warn'), val_pred, 'allow'):.1%}",
        "short val: allow blocked": f"{share(is_short & (y == 'allow'), val_pred, 'block'):.1%}",
        "own prompts correct": f"{(own_pred == expected).sum()} of {len(expected)}",
        "own: block prompts caught": f"{(own_pred[own_block] == 'block').sum()} of {own_block.sum()}",
        "own: casual block prompts caught": f"{(own_pred[own_block & casual] == 'block').sum()} of {(own_block & casual).sum()}",
        "own: harmless prompts blocked": f"{(own_pred[own_allow] == 'block').sum()} of {own_allow.sum()}",
        "own: warn prompts let through (allow)": f"{(own_pred[own_warn] == 'allow').sum()} of {own_warn.sum()}",
        "response time p95 (ms)": round(float(np.percentile(times, 95)), 1),
    }
    # metrics for MLflow, same names and definitions as the earlier candidate runs
    classic = evaluate(y[is_classic], val_pred[is_classic])
    metrics_for_mlflow[name] = {
        **{f"val_{k}": v for k, v in classic.items()},
        "val_harmless_to_block": share(is_harmless, val_pred, "block"),
        "val_harmless_to_warn": share(is_harmless, val_pred, "warn"),
        "val_de_recall_block": recall_score(y[is_classic & is_de], val_pred[is_classic & is_de],
                                            labels=["block"], average="macro"),
        "own_prompts_accuracy": float((own_pred == expected).mean()),
        "own_prompts_recall_block": share(own_block, own_pred, "block"),
        "own_prompts_casual_recall_block": share(own_block & casual, own_pred, "block"),
        "own_prompts_false_block_allow": share(own_allow, own_pred, "block"),
        "own_prompts_warn_to_allow": share(own_warn, own_pred, "allow"),
        "latency_p95_ms": float(np.percentile(times, 95)),
    }
    del bundle, pipe
    _ = gc.collect()

comparison = pd.DataFrame(columns)
display(comparison)

reports = PROJECT_DIR / "reports"
reports.mkdir(exist_ok=True)
comparison.to_csv(reports / "short_prompts_before_after.csv")
print("Saved: reports/short_prompts_before_after.csv")

,TF-IDF + harmless (before),TF-IDF + short prompts (after),SVM + harmless (before),SVM + short prompts (after)
block threshold,0.2,0.2,-0.4,-0.4
ai4privacy recall block,0.953,0.954,0.957,0.956
ai4privacy recall block (de),0.944,0.959,0.956,0.953
ai4privacy allow texts blocked,0.031,0.03,0.045,0.044
real harmless prompts allowed,93.2%,92.8%,91.4%,91.0%
real harmless prompts blocked,1.8%,1.6%,2.3%,2.2%
short val: correct,80.7%,93.0%,80.5%,90.3%
short val: block caught,97.5%,100.0%,100.0%,100.0%
short val: warn let through (allow),9.5%,3.0%,4.0%,2.0%
short val: allow blocked,15.5%,3.5%,17.0%,11.5%


Saved: reports/short_prompts_before_after.csv


In [8]:
# Log the two new models to MLflow (same parameter and metric names as the earlier candidates)
def common_params(model_name, C, threshold):
    return {"model": model_name, "analyzer": "char_wb", "ngram_range": "2-5", "min_df": 5,
            "max_features": 300_000, "C": C, "class_weight": "balanced",
            "block_threshold": "argmax" if threshold is None else threshold,
            "split_fingerprint": SPLIT_FINGERPRINT, "policy_version": POLICY_VERSION,
            "harmless_fingerprint": HARMLESS_FINGERPRINT, "short_prompts_fingerprint": SHORT_FINGERPRINT,
            "n_short_train": int((train["origin"] == "short").sum())}

for run_name, column, model_name, C, threshold, reason, converged, part in [
        ("tfidf_logreg_short", "TF-IDF + short prompts (after)", "tfidf_logreg", C_LOGREG, THRESHOLD_LR, reason_lr,
         converged_lr, "A_short_prompts"),
        ("svm_short", "SVM + short prompts (after)", "tfidf_linearsvc", C_SVM, THRESHOLD_SVM, reason_svm,
         converged_svm, "B_svm_short_data")]:
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({"milestone": "M2", "experiment_part": part, "candidate": "true"})
        mlflow.log_params({**common_params(model_name, C, threshold), "threshold_reason": reason})
        mlflow.log_metrics({**metrics_for_mlflow[column], "converged": int(converged)})
        mlflow.log_artifact(str(MODEL_DIR / f"{run_name}.joblib"))
        mlflow.log_artifact(str(reports / "short_prompts_before_after.csv"))
    print("Logged:", run_name)

🏃 View run tfidf_logreg_short at: https://dagshub.com/annihlj/AIOps_PromptGuard.mlflow/#/experiments/0/runs/1d583cce162c4607816f26c08496ef6e
🧪 View experiment at: https://dagshub.com/annihlj/AIOps_PromptGuard.mlflow/#/experiments/0
Logged: tfidf_logreg_short
🏃 View run svm_short at: https://dagshub.com/annihlj/AIOps_PromptGuard.mlflow/#/experiments/0/runs/7bae6e439db0423c90528af024a6abee
🧪 View experiment at: https://dagshub.com/annihlj/AIOps_PromptGuard.mlflow/#/experiments/0
Logged: svm_short


## 8. Own prompts the new models get wrong

For the team discussion. We only look at these, we do not change anything because of them.

In [9]:
for name in ["TF-IDF + short prompts (after)", "SVM + short prompts (after)"]:
    wrong = own.assign(predicted=own_predictions[name])
    wrong = wrong[wrong["predicted"] != wrong["expected_decision"]]
    print(f"{name}: wrong {len(wrong)} of {len(own)}")
    with pd.option_context("display.max_colwidth", 100, "display.max_rows", 300):
        display(wrong[["prompt", "expected_decision", "predicted", "category", "style"]])

TF-IDF + short prompts (after): wrong 30 of 200


,prompt,expected_decision,predicted,category,style
10,Bitte prüfe die Rechnungsnummer RE-2024-00817 auf Formatfehler.,allow,block,tricky number,formal
18,"bestellung 4500123987 hängt im sap, was tun?",allow,block,tricky number,casual
23,"schreib ne kurze antwort: danke, ich melde mich morgen",allow,warn,harmless,casual
32,was ist nochmal der unterschied zwischen pin und tan,allow,block,tricky keyword,casual
35,Wie lautet die korrekte Anrede in einem Brief an eine Behörde?,allow,warn,harmless,formal
37,"Unsere Ticketnummer ist INC0045821, schreib eine Statusmeldung an die Fachabteilung.",allow,warn,tricky number,formal
43,"Formuliere eine Antwort auf eine Kundenbeschwerde über eine verspätete Lieferung, ohne Namen.",allow,warn,harmless,formal
45,Artikelnummer 10-4471-B ist ausverkauft. Schreib einen Hinweis für den Onlineshop.,allow,warn,tricky number,formal
47,Schreibe eine Einladung zur Weihnachtsfeier am 18.12. ab 18 Uhr im Ratskeller.,allow,warn,harmless date/time,formal
53,"Hilf mir, ein Feedbackgespräch mit einem Teammitglied vorzubereiten.",allow,warn,harmless,formal


SVM + short prompts (after): wrong 34 of 200


,prompt,expected_decision,predicted,category,style
7,Wie erstelle ich ein sicheres Passwort? Ich brauche Tipps für eine Schulung.,allow,block,tricky keyword,formal
10,Bitte prüfe die Rechnungsnummer RE-2024-00817 auf Formatfehler.,allow,block,tricky number,formal
18,"bestellung 4500123987 hängt im sap, was tun?",allow,block,tricky number,casual
23,"schreib ne kurze antwort: danke, ich melde mich morgen",allow,warn,harmless,casual
32,was ist nochmal der unterschied zwischen pin und tan,allow,block,tricky keyword,casual
33,"Schreib eine Vorlage für eine Reisekostenabrechnung (Hotel, Bahn, Verpflegung).",allow,warn,harmless,formal
36,"Erstelle einen Wochenplan für die Schichten Früh, Spät und Nacht mit 6 Personen.",allow,warn,harmless numbers,formal
37,"Unsere Ticketnummer ist INC0045821, schreib eine Statusmeldung an die Fachabteilung.",allow,warn,tricky number,formal
43,"Formuliere eine Antwort auf eine Kundenbeschwerde über eine verspätete Lieferung, ohne Namen.",allow,warn,harmless,formal
45,Artikelnummer 10-4471-B ist ausverkauft. Schreib einen Hinweis für den Onlineshop.,allow,warn,tricky number,formal


## 9. Conclusion

All numbers are in the table of chapter 7.

**Did the data fix work?**
Yes. Warn prompts let through on our own prompts dropped from 20 to 7 of 60 (TF-IDF) and from 15 to 5 of 60
(SVM). Own prompts correct rose from 155 to 170 (TF-IDF) and from 156 to 166 of 200 (SVM). On the short-prompt
validation, warn prompts let through fell from 9.5 % to 3.0 % (TF-IDF) and from 4.0 % to 2.0 % (SVM).

**Did it cost anything?**
Almost nothing. Recall block on ai4privacy stays at 0.954 (TF-IDF) and 0.956 (SVM); for German, TF-IDF even rises
from 0.944 to 0.959 and now reaches the 95 % target too. Real harmless prompts allowed stay at 93 % (TF-IDF) and
91 % (SVM), and the response time stays at about 2.5 ms.

**Did the models learn the content or the frames?**
The content: the short-prompt validation uses sentence frames that were never in training, and the share of
correct decisions there rose from 80.7 % to 93.0 % (TF-IDF) and from 80.5 % to 90.3 % (SVM). Harmless short
prompts blocked fell from 15.5 % to 3.5 % (TF-IDF). Note that the block threshold was chosen on validation
including these prompts, so this number is slightly optimistic; the untouched short-prompt test set is used in
Milestone 5.

**Which models go on to Milestone 5**
TF-IDF + short prompts is now our strongest classical model: 170 of 200 own prompts correct, both targets reached
on validation, 0.959 recall block for German, and fewer false alarms than the SVM (5 vs. 10 of 80 own harmless
prompts blocked). SVM + short prompts stays as the alternative. The remaining errors are order and ticket numbers
that look like account numbers, questions about passwords, PINs or CVVs, and casually written credentials. These
need an understanding of the meaning, which is the argument for the encoder model.

**Limitations**
- The short prompts are generated from frames and only in German and English. Our own prompts stay the honest
  check, because they were written independently of the frames.
- The personal data values come from ai4privacy (synthetic). Values of validation and test come from the
  ai4privacy validation and test split, so no value is seen twice.